# Fase 1 — Gemini 5+1 · preparazione e preflight OFFLINE

Questo notebook **non invoca Gemini o Claude** e non avvia la campagna. Clona il branch sperimentale, verifica i dati e gli SHA, costruisce una **preview** del piano e, soltanto dopo attivazione esplicita, crea il piano in modalità **create-only** su Drive.

- V3 congelata: `6747eecc77c1b2fff7d581c7284578b5c574b4f7`
- Configurazione da studiare: `A1_8R_Planning`, `max_tool_rounds=6`, `max_verification_checks=1`.
- Otto run 5+1 pianificate; gli otto 4+2 restano bloccati finché non eseguiti i precedenti.

**Non inserire chiavi API in questo notebook.** La cella per le chiamate a pagamento non è inclusa. Se esistono file con claim incompleti, il preflight deve segnalarli, mai ripeterli.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

REPO = Path('/content/Tesi_OpenRCA')
DATA = Path('/content/drive/MyDrive/Tesi_OpenRCA')
BRANCH = 'feature/planner-budget-ablation-2026-11-05'
URL = 'https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git'

def git(*args):
    return subprocess.run(['git', '-C', str(REPO), *args],
                          check=True, text=True, capture_output=True).stdout.strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch',
                    BRANCH, URL, str(REPO)], check=True)
if git('branch', '--show-current') != BRANCH:
    raise RuntimeError('Branch non previsto: usare una working tree distinta e pulita.')
if git('status', '--porcelain'):
    raise RuntimeError('Repository non pulito. Non congelare il piano.')
HEAD = git('rev-parse', 'HEAD')
print('Branch:', BRANCH)
print('Commit esatto:', HEAD)

required = [
    DATA / 'data/processed/split_market_v1.csv',
    DATA / 'data/processed/benchmark_inputs_market.csv',
    DATA / 'data/protocols/v2/openrca_harness_v2_five_agents_development_v3_plan_v1.json',
]
for p in required:
    print(('OK' if p.is_file() else 'MANCA'), ':', p)
if not all(p.is_file() for p in required):
    raise FileNotFoundError('Mancano file necessari su Drive. Nessun piano creato.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO / 'requirements.txt')], check=True)
os.environ['PYTHONPATH'] = str(REPO / 'src')
sys.path.insert(0, str(REPO / 'src'))
try:
    print('langchain-google-genai:', version('langchain-google-genai'))
except PackageNotFoundError:
    print('Dipendenza Gemini assente: non avviare API.')
print('Bootstrap offline completato; nessuna inferenza LLM.')

## Verifica test e preview piano (zero API)

Questa cella valida le modifiche offline e ricostruisce in memoria il piano dagli otto incidenti del protocollo V3. **Non scrive nulla** nei risultati su Drive.

In [ ]:
from evaluation.planner_budget_ablation import build_plan, PROTOCOL
import json

test_env = {**os.environ, 'PYTHONPATH': str(REPO / 'src')}
subprocess.run([sys.executable, '-m', 'unittest', 'discover',
                '-s', str(REPO / 'src/tests'),
                '-p', 'test_planner_*.py', '-v'],
               cwd=REPO, env=test_env, check=True)

preview = build_plan(DATA, git('rev-parse', 'HEAD'))
assert preview['run_count'] == 16
assert preview['case_count'] == 8
assert all(c['arm_order'] == ['5p1', '4p2'] for c in preview['cases'])
print('Protocollo:', preview['protocol'])
print('Commit codice:', preview['git_commit'])
print('Piano V3 sorgente SHA-256:', preview['v3_plan_sha256'])
print('Modello:', preview['model_name'])
print('Budget massimo: 5 esplorazioni + 1 verifica')
print('Otto case_id 5+1:', [c['case_id'] for c in preview['cases']])
print('PREVIEW OFFLINE OK - Nessuna chiamata LLM, nessuna scrittura su Drive.')

## Congelamento del nuovo piano e preflight 5+1

**Controlla prima la preview.** Solo quando sei pronto a creare il nuovo piano, imposta `CREA_PIANO = True`. La creazione è **create-only**: non sovrascrive un piano esistente. Dopo aver creato il piano, questa cella verifica stati e provenienza senza inferenze.

Per la campagna a pagamento useremo un'altra procedura, dopo preventivo e autorizzazione esplicita.

In [ ]:
from evaluation.planner_budget_ablation import write_plan
from evaluation.planner_5p1_campaign import preflight_5p1

CREA_PIANO = False  # metti True SOLO dopo aver verificato la preview

p = DATA / 'data/protocols/v2' / f'{PROTOCOL}_plan_v1.json'
if CREA_PIANO:
    if p.exists():
        print('Piano già presente: nessuna sovrascrittura.')
    else:
        print('Nuovo piano create-only:', write_plan(DATA, REPO))
else:
    print('CREA_PIANO=False: nulla scritto su Drive.')

if p.exists():
    status = preflight_5p1(DATA, REPO)
    print(json.dumps(status, indent=2, ensure_ascii=False))
else:
    print('Preflight rimandato: il piano non è ancora stato creato.')